## Competition for nutrients in ENVIRONMENT (only compounds in metabolomics!!): consumption overlap

In [2]:
import pandas as pd
import json
import seaborn as sns
import numpy as np

import networkx as nx

from pyvis.network import Network

In [3]:
with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

In [4]:
results = pd.read_csv("../output/Venturelli2018/results_steadiercom/NONE_abundance.tsv",sep="\t")

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]

In [5]:
communities_12_exp_abundance = pd.read_csv("../data/Venturelli2018/generated/communities_12_exp_abundance.tsv",sep="\t",index_col=[0,1])
communities_12_exp_abundance

6
ER-  Blautia_hydrogenotrophica_DSM_10507     0.0109
     Collinsella_aerofaciens_ATCC_25986      0.0255
     Bacteroides_uniformis_ATCC_8492         0.3809
     Prevotella_copri_CB7_DSM_18205          0.0003
     Bacteroides_ovatus_ATCC_8483            0.1343
...                                             ...
NONE Eggerthella_lenta_DSM_2243              0.0168
     Faecalibacterium_prausnitzii_A2_165     0.2393
     Clostridium_hiranonis_TO_931_DSM_13275  0.0212
     Desulfovibrio_piger_ATCC_29098          0.0019
     Eubacterium_rectale_ATCC_33656          0.0086

[144 rows x 1 columns]

In [6]:
def centrality_metrics(network_data,cutoff=0,SP_centrality=True):
    """
    SP_centrality: centrality meassures based on shortest path
    """
    # Betweenness and closeness
    G = nx.Graph()
    G.add_nodes_from(list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique()))))
    
    if SP_centrality:
        edges = [(index[0],index[1],{"weight":1/data} ) for index,data in network_data.items() if data>cutoff]
    
    else:
        edges = [(index[0],index[1],{"weight":data} ) for index,data in network_data.items() if data>cutoff]
        
    G.add_edges_from(edges)
    
    if SP_centrality:
        return pd.DataFrame({"betweeness (1/data)":nx.centrality.betweenness_centrality(G,weight="weight"),
                     "closeness (1/data)":nx.centrality.closeness_centrality(G,distance="weight")})
    else:
        return pd.DataFrame({"degree (data)":dict(G.degree(weight="weight")),
                 "eigenvector (data)":nx.centrality.eigenvector_centrality(G,weight="weight")})


In [7]:
## From https://stackoverflow.com/questions/25668828/how-to-create-colour-gradient-in-python
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

def colorFader(c1,c2,mix=0): #fade (linear interpolate) from color c1 (at mix=0) to c2 (mix=1)
    c1=np.array(mpl.colors.to_rgb(c1))
    c2=np.array(mpl.colors.to_rgb(c2))
    return mpl.colors.to_hex((1-mix)*c1 + mix*c2)


In [8]:
b1='#f8cdcf' #bright pink
b2='#fa4f56' #red

c1='#ffffff' #white
c2='#0b5394' #blue

In [9]:
def spearman_corr_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network()

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    color = [colorFader(b1,b2,closeness_centrality_met[node]) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)
    
    
    maximum_corr = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value**2)*10,color=colorFader(c1,c2,value/maximum_corr)) 

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

In [10]:
def count_overlap_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network()
    
    maximum_centrality = pd.Series(closeness_centrality_met).max()

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    
    
    color = [colorFader(b1,b2,closeness_centrality_met[node]/maximum_centrality) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)

    maximum_overlap = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value**2)/10,color=colorFader(c1,c2,value/maximum_overlap))

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

### Spearman correlation

In [41]:
results_env = results[(results.donor=="environment")]
results_env = results_env[["receiver","compound","frequency"]]
# Find mean of frequencies
results_grouped_rec_env = results_env[(results_env.compound.isin(metid2metname.keys()))].groupby(["receiver","compound"]).mean()

**Correlation matrix**

In [42]:
network_data = results_grouped_rec_env.unstack().T.corr(method="spearman")
network_data = network_data.mask(np.tril(np.ones(network_data.shape)).astype(np.bool)).stack() 

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [43]:
centrality_metrics_SP = centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,0.583134
Eggerthella_lenta_DSM_2243,0.022222,0.622391
Desulfovibrio_piger_ATCC_29098,0.000000,0.650401
Bacteroides_vulgatus_ATCC_8482,0.000000,0.677135
Eubacterium_rectale_ATCC_33656,0.000000,0.698087
Clostridium_hiranonis_TO_931_DSM_13275,0.000000,0.726193
Blautia_hydrogenotrophica_DSM_10507,0.000000,0.739228
Bacteroides_ovatus_ATCC_8483,0.000000,0.750166
Faecalibacterium_prausnitzii_A2_165,0.000000,0.760608
Collinsella_aerofaciens_ATCC_25986,0.044444,0.771170


In [44]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

,degree (data),eigenvector (data)
Bacteroides_thetaiotaomicron_VPI_5482,5.788808,0.242946
Eggerthella_lenta_DSM_2243,5.852049,0.247598
Desulfovibrio_piger_ATCC_29098,6.120507,0.267387
Bacteroides_vulgatus_ATCC_8482,7.035124,0.298322
Clostridium_hiranonis_TO_931_DSM_13275,7.117623,0.300111
Eubacterium_rectale_ATCC_33656,7.301935,0.300298
Blautia_hydrogenotrophica_DSM_10507,7.738236,0.320753
Bacteroides_ovatus_ATCC_8483,7.826441,0.324547
Faecalibacterium_prausnitzii_A2_165,7.847945,0.324797
Collinsella_aerofaciens_ATCC_25986,8.079179,0.332325


In [45]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

In [46]:
net = spearman_corr_network(network_data,closeness_centrality_met=closeness_centrality_met)
net.show("example.html")

example.html


**Network topological properties (correlation>0.9!!!)**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [50]:
cutoff = 0.8

In [51]:
centrality_metrics_SP = centrality_metrics(network_data,cutoff=cutoff).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,0.503202
Eggerthella_lenta_DSM_2243,0.044444,0.559615
Desulfovibrio_piger_ATCC_29098,0.022222,0.560189
Clostridium_hiranonis_TO_931_DSM_13275,0.000000,0.589381
Eubacterium_rectale_ATCC_33656,0.022222,0.592370
Blautia_hydrogenotrophica_DSM_10507,0.044444,0.595730
Faecalibacterium_prausnitzii_A2_165,0.000000,0.629853
Bacteroides_ovatus_ATCC_8483,0.066667,0.638256
Bacteroides_vulgatus_ATCC_8482,0.000000,0.677135
Bacteroides_uniformis_ATCC_8492,0.177778,0.680769


In [52]:
centrality_metrics(network_data,SP_centrality=False,cutoff=0.9).sort_values("degree (data)")

,degree (data),eigenvector (data)
Desulfovibrio_piger_ATCC_29098,0.900234,0.250492
Bacteroides_vulgatus_ATCC_8482,0.901303,0.250790
Bacteroides_uniformis_ATCC_8492,0.917132,0.000015
Bacteroides_ovatus_ATCC_8483,0.927182,0.187251
Clostridium_hiranonis_TO_931_DSM_13275,0.929981,0.288499
Eubacterium_rectale_ATCC_33656,0.961538,0.000017
Faecalibacterium_prausnitzii_A2_165,1.851506,0.379208
Bacteroides_thetaiotaomicron_VPI_5482,1.917132,0.000026
Eggerthella_lenta_DSM_2243,1.961538,0.000027
Blautia_hydrogenotrophica_DSM_10507,2.710520,0.522464


**Network (cutoff correlation = 0.9 !!)**

In [53]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

In [54]:
net = spearman_corr_network(network_data,closeness_centrality_met=closeness_centrality_met,cutoff=cutoff)
net.show("example.html")

example.html


### Overlap count (compound consumed with high frequency)

In [22]:
results_env = results[(results.donor=="environment")]
results_env = results_env[["receiver","compound","frequency"]]
# Find mean of frequencies
results_grouped_rec_env = results_env[(results_env.compound.isin(metid2metname.keys()))].groupby(["receiver","compound"]).mean()

**Count (cutoff frequency=0.5!!)**

In [23]:
count_same = results_grouped_rec_env["frequency"]

count_same = count_same.unstack().fillna(0)>0.5
count_same = count_same.astype(int)
count_same = count_same.dot(count_same.T)

network_data = count_same.mask(np.tril(np.ones(count_same.shape)).astype(np.bool)).stack() 

maximum = network_data.max()

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [24]:
centrality_metrics_SP = centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Eggerthella_lenta_DSM_2243,0.000000,4.849885
Bacteroides_uniformis_ATCC_8492,0.000000,5.284127
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,6.290564
Desulfovibrio_piger_ATCC_29098,0.000000,6.400813
Blautia_hydrogenotrophica_DSM_10507,0.000000,6.444863
Bacteroides_vulgatus_ATCC_8482,0.000000,6.614173
Collinsella_aerofaciens_ATCC_25986,0.000000,6.845966
Bacteroides_ovatus_ATCC_8483,0.022222,7.558488
Clostridium_hiranonis_TO_931_DSM_13275,0.044444,7.610785
Faecalibacterium_prausnitzii_A2_165,0.000000,8.012487


In [25]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

,degree (data),eigenvector (data)
Eggerthella_lenta_DSM_2243,51.0,0.212501
Bacteroides_uniformis_ATCC_8492,56.0,0.229903
Bacteroides_thetaiotaomicron_VPI_5482,68.0,0.276355
Bacteroides_vulgatus_ATCC_8482,69.0,0.281894
Desulfovibrio_piger_ATCC_29098,72.0,0.300892
Blautia_hydrogenotrophica_DSM_10507,72.0,0.301857
Collinsella_aerofaciens_ATCC_25986,73.0,0.300516
Bacteroides_ovatus_ATCC_8483,78.0,0.310859
Clostridium_hiranonis_TO_931_DSM_13275,83.0,0.339456
Faecalibacterium_prausnitzii_A2_165,87.0,0.355232


**Network**

In [26]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = count_overlap_network(network_data,closeness_centrality_met,cutoff=0)
net.show("example.html")

example.html


**Cutoff=7!!**

In [27]:
centrality_metrics_SP = centrality_metrics(network_data,cutoff=7).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Eggerthella_lenta_DSM_2243,0.000000,0.000000
Bacteroides_uniformis_ATCC_8492,0.000000,3.687744
Bacteroides_vulgatus_ATCC_8482,0.000000,4.922004
Collinsella_aerofaciens_ATCC_25986,0.000000,5.071304
Bacteroides_thetaiotaomicron_VPI_5482,0.044444,5.089005
Blautia_hydrogenotrophica_DSM_10507,0.000000,5.236043
Desulfovibrio_piger_ATCC_29098,0.000000,5.244604
Bacteroides_ovatus_ATCC_8483,0.133333,5.735026
Clostridium_hiranonis_TO_931_DSM_13275,0.111111,6.319149
Faecalibacterium_prausnitzii_A2_165,0.088889,6.535833


In [28]:
centrality_metrics(network_data,SP_centrality=False,cutoff=7).sort_values("degree (data)")

,degree (data),eigenvector (data)
Eggerthella_lenta_DSM_2243,0.0,4.198209e-27
Bacteroides_uniformis_ATCC_8492,17.0,5.237260e-02
Bacteroides_vulgatus_ATCC_8482,26.0,1.281079e-01
Bacteroides_thetaiotaomicron_VPI_5482,35.0,1.270989e-01
Collinsella_aerofaciens_ATCC_25986,44.0,3.390709e-01
Bacteroides_ovatus_ATCC_8483,45.0,2.005441e-01
Desulfovibrio_piger_ATCC_29098,47.0,3.569281e-01
Blautia_hydrogenotrophica_DSM_10507,47.0,3.579933e-01
Clostridium_hiranonis_TO_931_DSM_13275,58.0,3.959824e-01
Faecalibacterium_prausnitzii_A2_165,62.0,4.280324e-01


In [30]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = count_overlap_network(network_data,closeness_centrality_met,cutoff=7)
net.show("example.html")

example.html
